# F08 - Regression and classification

**Hardware:** CPU-only · **Time:** 3 hours · **Prerequisite:** F07

Regression predicts a numeric value; classification predicts a class. This
notebook builds a house-price regressor and a student-pass classifier. It always
starts with a baseline, because a model that cannot beat "predict the mean" or
"predict the majority class" is not useful.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 42
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
print("pandas", pd.__version__)

pandas 2.3.3


## 1. Regression data and a baseline

The target is `sale_price`. The baseline predicts the training mean for every
house. Report MAE and RMSE: MAE is in dollars per house and easier to explain;
RMSE penalizes large errors more heavily.

In [2]:
def make_houses(n: int = 600, seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    area = rng.normal(1800, 500, n).clip(600, 4000)
    bedrooms = rng.integers(1, 6, n)
    age = rng.integers(0, 60, n)
    distance = rng.gamma(2, 2, n).clip(0.5, 25)
    garage = rng.integers(0, 2, n)
    hood = rng.choice(["north", "south", "east", "west"], n, p=[0.3, 0.25, 0.25, 0.2])
    effect = pd.Series(hood).map({"north": 35000, "south": -10000, "east": 15000, "west": -5000}).to_numpy()
    price = (
        120000 + 95 * area + 18000 * bedrooms - 900 * age - 4500 * distance
        + 12000 * garage + 0.00004 * area**2 + effect + rng.normal(0, 20000, n)
    ).clip(50000, None)
    return pd.DataFrame(
        {
            "area_sqft": area.round(1),
            "bedrooms": bedrooms,
            "age_years": age,
            "distance_km": distance.round(2),
            "garage": garage,
            "neighbourhood": hood,
            "sale_price": price.round(0),
        }
    )


houses = make_houses()
feature_columns = ["area_sqft", "bedrooms", "age_years", "distance_km", "garage"]
X = houses[feature_columns].to_numpy(dtype=float)
y = houses["sale_price"].to_numpy(dtype=float)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=SEED)
baseline = np.full_like(y_test, y_train.mean())
print("baseline MAE:", round(mean_absolute_error(y_test, baseline), 0))
print("baseline RMSE:", round(np.sqrt(mean_squared_error(y_test, baseline)), 0))
assert np.isfinite(baseline).all()

baseline MAE: 51734.0
baseline RMSE: 66281.0


## 2. Fit linear regression and ridge

`LinearRegression` minimizes squared error. `Ridge` adds a penalty on large
coefficients, which can stabilize estimates when features are correlated.
Standardize first so the penalty treats features comparably.

In [3]:
scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

linear = LinearRegression().fit(X_train_scaled, y_train)
ridge = Ridge(alpha=10.0).fit(X_train_scaled, y_train)

for name, model in {"linear": linear, "ridge": ridge}.items():
    prediction = model.predict(X_test_scaled)
    print(
        name,
        "MAE:", round(mean_absolute_error(y_test, prediction), 0),
        "RMSE:", round(np.sqrt(mean_squared_error(y_test, prediction)), 0),
        "R2:", round(r2_score(y_test, prediction), 3),
    )
    assert r2_score(y_test, prediction) > 0.5

linear MAE: 21708.0 RMSE: 27354.0 R2: 0.829
ridge MAE: 21699.0 RMSE: 27382.0 R2: 0.829


## 3. Inspect coefficients in plain units

The coefficients below are on the standardized scale, so they are comparable in
magnitude, not directly in dollars. A positive coefficient means the feature is
associated with a higher predicted price when the other features are held fixed.
Association is not causation.

In [4]:
coefficients = pd.Series(linear.coef_, index=feature_columns).sort_values(key=abs, ascending=False)

print(coefficients.round(1).to_string())
assert coefficients.abs().max() > 0

area_sqft      46020.4
bedrooms       25912.8
age_years     -14158.3
distance_km   -14062.4
garage          7128.6


## 4. Classification data and a majority baseline

The target is `passed`. The majority-class baseline predicts the most common
label. Accuracy alone can look good when one class dominates, so also report
precision, recall, and F1.

In [5]:
def make_students(n: int = 500, seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    study = rng.gamma(2.0, 2.0, n).clip(0, 20)
    attendance = rng.uniform(40, 100, n)
    prior = rng.normal(65, 12, n).clip(0, 100)
    sleep = rng.normal(7, 1.2, n).clip(3, 10)
    tutoring = rng.choice(["yes", "no"], n, p=[0.35, 0.65])
    course = rng.choice(["Math", "Physics", "CS"], n, p=[0.4, 0.3, 0.3])
    logit = (
        -11.5 + 0.30 * study + 0.055 * attendance + 0.075 * prior
        + 0.20 * sleep + 1.1 * (tutoring == "yes")
    )
    passed = rng.binomial(1, 1 / (1 + np.exp(-logit)))
    return pd.DataFrame(
        {
            "study_hours": study.round(2),
            "attendance": attendance.round(1),
            "prior_score": prior.round(1),
            "sleep_hours": sleep.round(2),
            "tutoring": tutoring,
            "course": course,
            "passed": passed,
        }
    )


students = make_students()
X_cat = pd.get_dummies(students[["tutoring", "course"]], drop_first=True, dtype=float)
X_num = students[["study_hours", "attendance", "prior_score", "sleep_hours"]]
X_cls = pd.concat([X_num, X_cat], axis=1).to_numpy(dtype=float)
y_cls = students["passed"].to_numpy()
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_cls, y_cls, test_size=0.25, random_state=SEED, stratify=y_cls
)
majority = int(np.bincount(y_train_c).argmax())
baseline_pred = np.full_like(y_test_c, majority)
print("majority class:", majority, "baseline accuracy:", round(accuracy_score(y_test_c, baseline_pred), 3))
print("class balance:", np.bincount(y_train_c) / len(y_train_c))
assert set(np.unique(y_train_c)) == {0, 1}

majority class: 0 baseline accuracy: 0.512
class balance: [0.50933333 0.49066667]


## 5. Logistic regression and the confusion matrix

Logistic regression models the log-odds of the positive class. The confusion
matrix counts true negatives, false positives, false negatives, and true
positives. Read it before reading a single aggregate metric.

In [6]:
scaler_c = StandardScaler().fit(X_train_c)
classifier = LogisticRegression(max_iter=2000, random_state=SEED).fit(
    scaler_c.transform(X_train_c), y_train_c
)
pred = classifier.predict(scaler_c.transform(X_test_c))
matrix = confusion_matrix(y_test_c, pred)
print("confusion matrix [[TN, FP], [FN, TP]]:")
print(matrix)
print("accuracy:", round(accuracy_score(y_test_c, pred), 3))
print("precision:", round(precision_score(y_test_c, pred), 3))
print("recall:", round(recall_score(y_test_c, pred), 3))
print("F1:", round(f1_score(y_test_c, pred), 3))
print("majority baseline:", round(accuracy_score(y_test_c, baseline_pred), 3))
assert accuracy_score(y_test_c, pred) > accuracy_score(y_test_c, baseline_pred)

confusion matrix [[TN, FP], [FN, TP]]:
[[51 13]
 [21 40]]
accuracy: 0.728
precision: 0.755
recall: 0.656
F1: 0.702
majority baseline: 0.512


## Which metric should you report?

| Decision context | Prefer | Why |
| --- | --- | --- |
| Missing a positive case is costly | Recall | You want few false negatives |
| False alarms are costly | Precision | You want few false positives |
| Balance precision and recall | F1 | One number, but hides the trade-off |
| Ranking is enough | ROC-AUC or PR-AUC | Threshold-free comparison |
| Dollar error matters | MAE or RMSE | Interpretable regression error |

A metric is a claim about what matters. State the decision and the cost of each
error before choosing it.

**Next:** F09 - Decision trees, random forests, and gradient boosting.